# Global Citizen (GC) Scoring Pipeline
**Population:** 199 Students — `FULL_199_students_MASTER_SHEET.csv`

---
### Dimensions & Weights
| Dimension | Weight | Columns |
|---|---|---|
| D1 — Empathy & Humility | 30% | 10 Likert 1–7 (pre + post) |
| D2 — Community Feel | 20% | 2 Likert 1–7 + Culture Feel (lookup map) |
| D3 — Cultural Competency | 20% | 1 Likert 1–7 + 2 Binary |
| D4 — Network & Growth | 20% | 4 Binary + 1 Count |
| D5 — Volunteering | 10% | 1 Binary + 1 Hours range |

In [1]:
# ── Cell 1 — Setup & Load ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

INPUT_FILE  = 'FULL_199_students_MASTER_SHEET.csv'
OUTPUT_FILE = 'GC_Scores_199_Students.csv'

df = pd.read_csv(INPUT_FILE, encoding='latin1')
print(f'Loaded {len(df)} students, {df.shape[1]} columns')
df.head(3)

Loaded 199 students, 101 columns


,Student Name,Pre Empathy,Pre Humble,Pre Listen,Pre Include Others Who Are Different,Pre Deal with Conflicts,Pre Lead With Authenticity,Listen to others,Deal with conflicts with other people conflict management,Include others who are different from you diversity and inclusion,...,Rate your Career Pathways Champion (1-10),Did you feel your Champion was caring and supportive,Did you find a way to stay in touch,How did the SPEAKHIRE team do with the match,I would recommend the SPEAKHIRE Foundational Year to my friends,Were you introduced to diverse career professionals,Listen to others.1,Deal with conflicts - conflict management,Include others who are different - diversity and inclusion,Did you learn something about other careers from other Career Cohorts
0,Angel Tavarez,6,6,6,7,6,4,5,6,7,...,10.0,1,"No, because I don't have time, I have other th...",8,7.00,1,5,6,7,yes
1,Aristid Reynoso,6,7,6,6,6,7,7,7,6,...,10.0,1,"No, because I don't have time, I have other th...",8,5.00,1,7,7,6,Yes
2,Ayele Dounou,6,7,6,5,6,4,5,5,7,...,10.0,1,"Yes, we exchanged contact information at the e...",8,5.67,1,5,5,7,Yea


In [2]:
# ── Cell 2 — Normalisation Helpers ────────────────────────────────────────

def norm_1_7(series):
    """Likert 1–7 → 0–1"""
    s = pd.to_numeric(series, errors='coerce')
    return ((s - 1) / 6).clip(0, 1)

def norm_binary(series):
    """TRUE/FALSE/1/0 → 0–1"""
    s_str = series.astype(str).str.strip().str.upper()
    result = pd.to_numeric(series, errors='coerce').clip(0, 1)
    result[s_str.isin(['YES', 'TRUE'])]  = 1.0
    result[s_str.isin(['NO',  'FALSE'])] = 0.0
    return result

def norm_minmax(series):
    """Count/numeric → 0–1 via min-max"""
    s = pd.to_numeric(series, errors='coerce')
    mn, mx = s.min(), s.max()
    if mx == mn:
        return s.where(s.isna(), 0.5)
    return ((s - mn) / (mx - mn)).clip(0, 1)

def norm_hours_volunteered(series):
    """
    Hours volunteered bins → 0–1
    Handles both numeric ('21') and range strings ('20-30')
    """
    def parse_val(v):
        v = str(v).strip()
        if v in ('', 'nan'): return np.nan
        try: num = float(v)
        except ValueError:
            try: num = float(v.split('-')[0])
            except: return np.nan
        if num == 0:  return 0.0
        if num < 10:  return 0.1
        if num < 20:  return 0.3
        if num < 30:  return 0.5
        if num < 60:  return 0.7
        return 1.0
    return series.apply(parse_val)

# ── Culture Feel: direct lookup map ────────────────────────────────────────
# 10 unique values across all 199 students — no LLM needed
# Text responses: score reflects clarity and positivity of cultural identity
# Numeric responses (2–7): treat as Likert → rescale (val-1)/6
CULTURE_FEEL_MAP = {
    'I feel equally of both':                        1.00,  # strongest — dual identity embraced
    'I feel more of the culture of my country of origin': 0.7,  # clear identity, single-culture leaning
    'I feel more American':                          0.6,  # clear identity, single-culture leaning
    'I am confused about my culture':                0.25,  # uncertainty / conflict
    '7': (7-1)/6, '6': (6-1)/6, '5': (5-1)/6,
    '4': (4-1)/6, '3': (3-1)/6, '2': (2-1)/6,
}

def norm_culture_feel(series):
    return series.astype(str).str.strip().map(CULTURE_FEEL_MAP).fillna(0.5)

print('Normalisation helpers defined ✓')
print('\nCulture Feel lookup map:')
for k, v in CULTURE_FEEL_MAP.items():
    print(f'  {k:<55} → {v:.3f}')

Normalisation helpers defined ✓

Culture Feel lookup map:
  I feel equally of both                                  → 1.000
  I feel more of the culture of my country of origin      → 0.700
  I feel more American                                    → 0.600
  I am confused about my culture                          → 0.250
  7                                                       → 1.000
  6                                                       → 0.833
  5                                                       → 0.667
  4                                                       → 0.500
  3                                                       → 0.333
  2                                                       → 0.167


In [3]:
# ── Cell 3 — Apply Normalisations ─────────────────────────────────────────

# D1 — Empathy & Humility
df['n_pre_empathy']   = norm_1_7(df['Pre Empathy'])
df['n_pre_humble']    = norm_1_7(df['Pre Humble'])
df['n_pre_listen']    = norm_1_7(df['Pre Listen'])
df['n_pre_include']   = norm_1_7(df['Pre Include Others Who Are Different'])
df['n_pre_conflict']  = norm_1_7(df['Pre Deal with Conflicts'])
df['n_pre_auth']      = norm_1_7(df['Pre Lead With Authenticity'])
df['n_post_listen']   = norm_1_7(df['Listen to others'])
df['n_post_conflict'] = norm_1_7(df['Deal with conflicts with other people conflict management'])
df['n_post_include']  = norm_1_7(df['Include others who are different from you diversity and inclusion'])
df['n_post_reflect']  = norm_1_7(df['Reflect if you have been in a similar situation as someone you are trying to help non positional leadership'])

# D2 — Community
df['n_community_feel'] = norm_1_7(df['Community Feel (Quant)'])
df['n_pre_community']  = norm_1_7(df['Pre Community Connected'])
df['n_culture_feel']   = norm_culture_feel(df['Culture Feel'])

# D3 — Cultural Competency
df['n_cultural_values'] = norm_1_7(df['I understand how my cultural values can shape my career choices'])
df['n_diverse_peers']   = norm_binary(df['After meeting Champions and working with other peers in my SPEAKHIRE Internship Rounds I better understand people who are different from me'])
df['n_diverse_prof']    = norm_binary(df['Were you introduced to diverse career professionals who you can relate to during this Internship Round'])

# D4 — Network & Growth
df['n_belong']         = norm_binary(df['Meeting with my Champions during school helped me feel like I belong in school'])
df['n_network_value']  = norm_binary(df['This SPEAKHIRE Foundational Year helped me understand the value of building a strong network'])
df['n_engaged']        = norm_binary(df['I feel more engaged in school and participate more than before'])
df['n_new_friends']    = norm_binary(df['I made new friends during the Foundational Year'])
df['n_career_network'] = norm_minmax(df['How many individuals do you know who work in the career you are interested in'])

# D5 — Volunteering
df['n_ever_volunteered']  = norm_binary(df['FY1 Ever Volunteered'])
df['n_hours_volunteered'] = norm_hours_volunteered(df['FY1 Hours Volunteered'])

print('All 23 columns normalised ✓')
check_cols = [
    'n_pre_empathy','n_post_reflect','n_community_feel',
    'n_culture_feel','n_cultural_values','n_belong',
    'n_ever_volunteered','n_hours_volunteered'
]
print(df[check_cols].describe().round(3))

All 23 columns normalised ✓
       n_pre_empathy  n_post_reflect  n_community_feel  n_culture_feel  \
count        199.000         199.000           199.000         199.000   
mean           0.721           0.771             0.677           0.764   
std            0.242           0.188             0.212           0.205   
min            0.000           0.333             0.000           0.167   
25%            0.667           0.667             0.500           0.700   
50%            0.833           0.833             0.667           0.700   
75%            0.833           0.917             0.833           1.000   
max            1.000           1.000             1.000           1.000   

       n_cultural_values  n_belong  n_ever_volunteered  n_hours_volunteered  
count            199.000   199.000             199.000              199.000  
mean               0.791     0.658               0.462                0.216  
std                0.211     0.475               0.500                0

In [4]:
# ── Cell 4 — Dimension Sub-Scores ─────────────────────────────────────────

# D1 — Empathy & Humility (30%)
# Pre = 30% of dimension weight, Post = 70%
df['d1_empathy'] = (
    df['n_pre_empathy']   * 0.05 +
    df['n_pre_humble']    * 0.05 +
    df['n_pre_listen']    * 0.05 +
    df['n_pre_include']   * 0.05 +
    df['n_pre_conflict']  * 0.05 +
    df['n_pre_auth']      * 0.05 +
    df['n_post_listen']   * 0.15 +
    df['n_post_conflict'] * 0.15 +
    df['n_post_include']  * 0.15 +
    df['n_post_reflect']  * 0.20
)

# D2 — Community Feel (20%)
df['d2_community'] = (
    df['n_pre_community']  * 0.15 +
    df['n_community_feel'] * 0.40 +
    df['n_culture_feel']   * 0.45
)

# D3 — Cultural Competency (20%)
df['d3_cultural'] = (
    df['n_cultural_values'] * 0.30 +
    df['n_diverse_peers']   * 0.30 +
    df['n_diverse_prof']    * 0.40
)

# D4 — Network & Growth (20%)
df['d4_network'] = (
    df['n_belong']         * 0.15 +
    df['n_network_value']  * 0.15 +
    df['n_engaged']        * 0.15 +
    df['n_new_friends']    * 0.15 +
    df['n_career_network'] * 0.40
)

# D5 — Volunteering (10%)
df['d5_volunteering'] = (
    df['n_ever_volunteered']  * 0.35 +
    df['n_hours_volunteered'] * 0.65
)

dims = ['d1_empathy','d2_community','d3_cultural','d4_network','d5_volunteering']
print('Dimension sub-scores ✓')
print(df[dims].describe().round(3))

Dimension sub-scores ✓
       d1_empathy  d2_community  d3_cultural  d4_network  d5_volunteering
count     199.000       199.000      199.000     199.000          199.000
mean        0.704         0.724        0.810       0.607            0.302
std         0.115         0.136        0.209       0.183            0.344
min         0.333         0.279        0.100       0.114            0.000
25%         0.633         0.635        0.650       0.471            0.000
50%         0.717         0.732        0.900       0.621            0.000
75%         0.779         0.817        1.000       0.714            0.675
max         0.942         1.000        1.000       1.000            1.000


In [5]:
# ── Cell 5 — Final GC Score (0–100) & Percentile ──────────────────────────

df['gc_score'] = (
    df['d1_empathy']      * 0.30 +
    df['d2_community']    * 0.20 +
    df['d3_cultural']     * 0.20 +
    df['d4_network']      * 0.20 +
    df['d5_volunteering'] * 0.10
) * 100

df['gc_score']      = df['gc_score'].round(1)
df['gc_percentile'] = (df['gc_score'].rank(pct=True) * 100).round(0).astype(int)

# Scale dimensions to 0–100 for readability
for d in ['d1_empathy','d2_community','d3_cultural','d4_network','d5_volunteering']:
    df[f'{d}_100'] = (df[d] * 100).round(1)

print('=' * 70)
print(f'GLOBAL CITIZEN SCORES -- ALL {len(df)} STUDENTS')
print('=' * 70)

display_cols = [
    'Student Name',
    'd1_empathy_100','d2_community_100','d3_cultural_100',
    'd4_network_100','d5_volunteering_100',
    'gc_score','gc_percentile'
]
rename_map = {
    'd1_empathy_100':      'Empathy',
    'd2_community_100':    'Community',
    'd3_cultural_100':     'Cultural',
    'd4_network_100':      'Network',
    'd5_volunteering_100': 'Volunteer',
    'gc_score':            'GC Score',
    'gc_percentile':       'Percentile',
}
print(df[display_cols].rename(columns=rename_map).to_string(index=False))

gc = df['gc_score']
print(f'\nCohort mean:    {gc.mean():.1f}')
print(f'Cohort median:  {gc.median():.1f}')
print(f'Min:            {gc.min():.1f}  ({df.loc[gc.idxmin(), "Student Name"]})')
print(f'Max:            {gc.max():.1f}  ({df.loc[gc.idxmax(), "Student Name"]})')
print(f'\nScore distribution:')
print(pd.cut(gc, bins=[0,20,40,60,80,100]).value_counts().sort_index())

GLOBAL CITIZEN SCORES -- ALL 199 STUDENTS
                 Student Name  Empathy  Community  Cultural  Network  Volunteer  GC Score  Percentile
                Angel Tavarez     78.3       59.5     100.0     65.7        0.0      68.5          53
              Aristid Reynoso     89.2       68.7      85.0     30.0       80.5      71.5          66
                 Ayele Dounou     71.7       61.5     100.0     65.7       80.5      75.0          78
                Emanuel Ortiz     75.8       68.2      95.0     30.0        0.0      61.4          25
            Ephraim Orisakiya     51.7       59.5      55.0     30.0       80.5      52.4          10
                 Ismatu Barry     86.7       73.2     100.0     65.7       67.5      80.5          90
                   Jaime Lara     82.5       68.2     100.0     77.1        0.0      73.8          74
               Jason Martinez     60.8       79.2      85.0     35.7       80.5      66.3          45
                  Jayde Smoak     71.7  

In [6]:
# ── Cell 6 — Save CSV & Download ──────────────────────────────────────────

output_cols = [
    'Student Name',
    'd1_empathy_100','d2_community_100','d3_cultural_100',
    'd4_network_100','d5_volunteering_100',
    'gc_score','gc_percentile',
    # Raw normalised inputs for audit
    'n_pre_empathy','n_pre_humble','n_pre_listen','n_pre_include',
    'n_pre_conflict','n_pre_auth',
    'n_post_listen','n_post_conflict','n_post_include','n_post_reflect',
    'n_community_feel','n_pre_community','n_culture_feel',
    'n_cultural_values','n_diverse_peers','n_diverse_prof',
    'n_belong','n_network_value','n_engaged','n_new_friends','n_career_network',
    'n_ever_volunteered','n_hours_volunteered',
]

df[output_cols].rename(columns={
    'd1_empathy_100':      'D1_Empathy_Humility',
    'd2_community_100':    'D2_Community',
    'd3_cultural_100':     'D3_Cultural_Competency',
    'd4_network_100':      'D4_Network_Growth',
    'd5_volunteering_100': 'D5_Volunteering',
    'gc_score':            'GC_Score',
    'gc_percentile':       'GC_Percentile',
}).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print(f'Output saved: {OUTPUT_FILE}')
print(f'  {len(df)} students | {len(output_cols)} columns')

# Download in Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print('Download triggered ✓')
except ImportError:
    print(f'(Not in Colab — file saved at: {OUTPUT_FILE})')

print('\n' + '='*50)
print('  GLOBAL CITIZEN SCORING PIPELINE -- COMPLETE')
print('='*50)

Output saved: GC_Scores_199_Students.csv
  199 students | 31 columns


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered ✓

  GLOBAL CITIZEN SCORING PIPELINE -- COMPLETE
